# Step 4: Hybrid Retrieval & Re-ranking Engine
### Project: Consumer Financial Complaints Intelligence System
**Goal:** Build a state-of-the-art hybrid search system that combines:
1. **Sparse Lexical Search (BM25):** For exact keyword matching (statutes, account numbers, specific fees).
2. **Dense Semantic Search (FAISS + Sentence-Transformers):** For intent and conceptual similarity.
3. **Reciprocal Rank Fusion (RRF):** Mathematical fusion algorithm to combine sparse & dense rank lists.
4. **Cross-Encoder Re-Ranking:** Deep semantic scoring on top candidates for maximum precision.

In [15]:
import os
import json
import time
import numpy as np
import pandas as pd

# 1. Sparse Lexical Search
from rank_bm25 import BM25Okapi

# 2. Dense Semantic Search & Cross-Encoder
from sentence_transformers import SentenceTransformer, CrossEncoder

# 3. Vector Database
import faiss

print(" All Hybrid Retrieval & RAG libraries imported successfully!")

 All Hybrid Retrieval & RAG libraries imported successfully!


In [20]:
from sklearn.model_selection import train_test_split

data_path = os.path.join('data', 'processed', 'train.parquet')
train_df = pd.read_parquet(data_path)

# 1. Label mapping load karte hain
with open(os.path.join('data', 'processed', 'label_mapping.json'), 'r') as f:
    label_mapping = {int(k): v for k, v in json.load(f).items()}

# 2. 10,000 complaints ka clean stratified sample (Is se 'label' column kabhi gayab nahi hoga!)
SAMPLE_SIZE = 10000
corpus_df, _ = train_test_split(
    train_df, 
    train_size=SAMPLE_SIZE, 
    stratify=train_df['label'], 
    random_state=42
)
corpus_df = corpus_df.reset_index(drop=True)

# 3. Text aur Labels list banate hain
corpus_texts = corpus_df['text'].tolist()
corpus_labels = [label_mapping[lbl] for lbl in corpus_df['label'].tolist()]

print("--- Knowledge Base Corpus Loaded Successfully! ---")
print(f"Total Historical Complaints Indexed: {len(corpus_texts):,}")
print(f"Categories in Knowledge Base       : {len(set(corpus_labels))}")
print("\nSample Complaint in Knowledge Base:")
print(f"[{corpus_labels[0].upper()}]: {corpus_texts[0][:120]}...")

--- Knowledge Base Corpus Loaded Successfully! ---
Total Historical Complaints Indexed: 10,000
Categories in Knowledge Base       : 7

Sample Complaint in Knowledge Base:
[CREDIT REPORTING]: my credit report is showing an account for acct that has been paid in full which when the account was paid in full agree...


## 1. Sparse Lexical Search Index (BM25)
Building a BM25 index using the Okapi BM25 ranking algorithm:
- Normalizes document lengths using $b=0.75$ to penalize bloated narratives.
- Saturates term frequency using $k_1=1.5$ to prevent repetitive keyword hijacking.
- Specializes in exact keyword matching for financial terms, statutes, and account codes.

In [ ]:
# 1. 10,000 complaints ko words me tokenize (split) karte hain
print("Tokenizing corpus for BM25...")
tokenized_corpus = [doc.lower().split() for doc in corpus_texts]

# 2. BM25 Index build karte hain (Okapi BM25 algorithm)
start_time = time.time()
bm25 = BM25Okapi(tokenized_corpus)
print(f" BM25 Index built successfully in {time.time() - start_time:.2f} seconds!")

# 3. BM25 Search Function
def search_bm25(query, top_k=5):
    # lowercase and split the query into words
    tokenized_query = query.lower().split()
    
    # calculate BM25 scores for the query against the corpus
    scores = bm25.get_scores(tokenized_query)
    
    # Top K highest scoring documents
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            'rank': rank,
            'corpus_idx': idx,
            'score': round(float(scores[idx]), 3),
            'category': corpus_labels[idx],
            'text': corpus_texts[idx]
        })
    return results

# Test Run on a exact financial query:
test_query = "escrow analysis refund check not received"
bm25_results = search_bm25(test_query, top_k=3)

print(f"\n BM25 Search Results for Query: \"{test_query}\"\n")
for res in bm25_results:
    print(f"Rank {res['rank']} [Score: {res['score']}] - Category: {res['category'].upper()}")
    print(f"   Snippet: \"{res['text'][:140]}...\"\n")

Tokenizing corpus for BM25...
 BM25 Index built successfully in 0.38 seconds!

 BM25 Search Results for Query: "escrow analysis refund check not received"

Rank 1 [Score: 20.854] - Category: MORTGAGE
   Snippet: "i refinanced my mortgage with which i had to make escrow payments with a different company the mortgage was serviced through another provide..."

Rank 2 [Score: 20.571] - Category: MORTGAGE
   Snippet: "my mortgage co is loancare an fha lender i bought a home that was taxed at a much higher rate then the home is now worth i have not missed a..."

Rank 3 [Score: 19.676] - Category: MORTGAGE
   Snippet: "ocwen did an escrow analysis and decided to increase escrow payment per month payment increase notification was not received so next month p..."



## 2. Dense Semantic Search Index (Sentence-Transformers + FAISS)
Building a dense vector retrieval pipeline to capture semantic intent and conceptual similarity:
- **Embedding Model:** `all-MiniLM-L6-v2` (384-dimensional dense vectors).
- **Vector Database Index:** Meta's `faiss.IndexFlatIP` (Inner Product) on L2-normalized vectors for exact Cosine Similarity.
- **Strength:** Captures synonyms, paraphrasing, and semantic intent even when exact keywords do not overlap.

In [ ]:
# 1. Embedding Model load karte hain
print("Loading Sentence-Transformer model ('all-MiniLM-L6-v2')...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. 10,000 complaints ko 384-dimensional vectors me convert karte hain
print("Encoding 10,000 complaints into dense vectors (takes ~30-40 seconds)...")
start_time = time.time()
corpus_embeddings = embedding_model.encode(
    corpus_texts, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True  # L2 Normalize karte hain taaki Dot Product = Cosine Similarity ban jaye!
)
print(f"-- Encoding completed in {time.time() - start_time:.2f} seconds!")
print(f"Embeddings Shape: {corpus_embeddings.shape}")

# 3. FAISS Vector Database Index banate hain
embedding_dim = corpus_embeddings.shape[1]  # 384
faiss_index = faiss.IndexFlatIP(embedding_dim)  # Flat Inner Product (Exact Cosine Similarity)

# Vectors ko FAISS index me add karte hain
faiss_index.add(corpus_embeddings.astype('float32'))
print(f"-> FAISS Index ready with {faiss_index.ntotal:,} vectors!")

# 4. FAISS Semantic Search Function
def search_faiss(query, top_k=5):
    # Normalize the query and get its vector
    query_vec = embedding_model.encode([query], normalize_embeddings=True).astype('float32')
    
    # FAISS me microsecond search
    scores, indices = faiss_index.search(query_vec, top_k)
    
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        results.append({
            'rank': rank,
            'corpus_idx': int(idx),
            'score': round(float(score), 3),
            'category': corpus_labels[idx],
            'text': corpus_texts[idx]
        })
    return results

# Test Run: Ek semantic query jisme "Mortgage" ya "Foreclosure" keyword NAHI hai:
semantic_query = "bank sold my property unlawfully without giving any prior alert"
faiss_results = search_faiss(semantic_query, top_k=3)

print(f"\n FAISS Semantic Results for Query: \"{semantic_query}\"\n")
for res in faiss_results:
    print(f"Rank {res['rank']} [Cosine Score: {res['score']}] - Category: {res['category'].upper()}")
    print(f"   Snippet: \"{res['text'][:140]}...\"\n")
    

Loading Sentence-Transformer model ('all-MiniLM-L6-v2')...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5652.40it/s]


Encoding 10,000 complaints into dense vectors (takes ~30-40 seconds)...


Batches: 100%|██████████| 157/157 [02:30<00:00,  1.05it/s]

-- Encoding completed in 150.17 seconds!
Embeddings Shape: (10000, 384)
-> FAISS Index ready with 10,000 vectors!

 FAISS Semantic Results for Query: "bank sold my property unlawfully without giving any prior alert"

Rank 1 [Cosine Score: 0.644] - Category: MORTGAGE
   Snippet: "i was not notified by my mortgage company wells fargo that my loan was sold to another company the law states i need to be notified days bef..."

Rank 2 [Cosine Score: 0.618] - Category: MORTGAGE
   Snippet: "bank sold my mortgage and did n t notify me in a timely manner usbank bought my mortgage and did not notify me in a timely manner..."

Rank 3 [Cosine Score: 0.584] - Category: MORTGAGE
   Snippet: "i was under review with bank of america loan modification and was never informed of my house going to sale until when i got and eviction not..."



## 3. Hybrid Retrieval with Reciprocal Rank Fusion (RRF)
Merging sparse lexical search (BM25) and dense semantic search (FAISS) using Reciprocal Rank Fusion:
- **Formula:** $RRF(d) = \sum_{m \in \{BM25, FAISS\}} \frac{1}{k + r_m(d)}$ where $k=60$.
- **Why RRF?** Eliminates the need for complex score calibration and normalizes heterogeneous score distributions purely based on ordinal ranks.

In [26]:
def hybrid_search_rrf(query, top_k=5, candidate_pool=20, k_constant=60):
    # 1. top 20 candidates from BM25
    bm25_candidates = search_bm25(query, top_k=candidate_pool)
    
    # 2. FAISS se top 20 candidates nikaalte hain
    faiss_candidates = search_faiss(query, top_k=candidate_pool)
    
    # 3. RRF Scores calculate karte hain
    rrf_scores = {}
    
    # BM25 ke ranks process karte hain
    for item in bm25_candidates:
        idx = item['corpus_idx']
        rank = item['rank']
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_constant + rank))
        
    # FAISS ke ranks process karte hain
    for item in faiss_candidates:
        idx = item['corpus_idx']
        rank = item['rank']
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_constant + rank))
        
    # 4. RRF scores ke hisab se descending order me sort karte hain
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    
    # 5. Final Hybrid Results list tayyar karte hain
    hybrid_results = []
    for rank, idx in enumerate(sorted_indices, 1):
        hybrid_results.append({
            'rank': rank,
            'corpus_idx': idx,
            'rrf_score': round(rrf_scores[idx], 5),
            'category': corpus_labels[idx],
            'text': corpus_texts[idx]
        })
    return hybrid_results

# Test Run: Ek complex realistic query
test_hybrid_query = "unauthorized transactions and fraudulent credit card fees disputed but denied"
hybrid_results = hybrid_search_rrf(test_hybrid_query, top_k=3)

print(f"--- HYBRID SEARCH (RRF) RESULTS for: \"{test_hybrid_query}\" ---\n")
for res in hybrid_results:
    print(f"Rank {res['rank']} [RRF Score: {res['rrf_score']}] - Category: {res['category'].upper()}")
    print(f"   Snippet: \"{res['text'][:150]}...\"\n")

--- HYBRID SEARCH (RRF) RESULTS for: "unauthorized transactions and fraudulent credit card fees disputed but denied" ---

Rank 1 [RRF Score: 0.03252] - Category: BANK ACCOUNT OR SERVICE
   Snippet: "my debit card was lost stolen and many unauthorized transactions appeared on my account i disputed these charges and they were denied i called suntrus..."

Rank 2 [RRF Score: 0.02942] - Category: BANK ACCOUNT OR SERVICE
   Snippet: "i disputed fraudulent transactions on my chase debit card they initially credited and to my checking account then reversed the credit they gave no exp..."

Rank 3 [RRF Score: 0.02727] - Category: CREDIT CARD
   Snippet: "i lost my credit card and and their are fraudulent charges and transactions of the charges do n t belong to me i have never used my credit card..."



In [28]:
# Check karte hain ki akele BM25 ne kya socha tha aur akele FAISS ne kya socha tha:
print("---  BM25 ke Top 3 Results ---")
for r in search_bm25(test_hybrid_query, top_k=3):
    print(f"Rank {r['rank']} [{r['category']}]: {r['text'][:90]}...")

print("\n--- FAISS ke Top 3 Results ---")
for r in search_faiss(test_hybrid_query, top_k=3):
    print(f"Rank {r['rank']} [{r['category']}]: {r['text'][:90]}...")

---  BM25 ke Top 3 Results ---
Rank 1 [Credit reporting]: i m a victim of identity theft my credit card has been lost stolen and compromised all of ...
Rank 2 [Bank account or service]: my debit card was lost stolen and many unauthorized transactions appeared on my account i ...
Rank 3 [Credit reporting]: i am a victim of identity theft and have been a victim of identity theft for over years th...

--- FAISS ke Top 3 Results ---
Rank 1 [Bank account or service]: my debit card was lost stolen and many unauthorized transactions appeared on my account i ...
Rank 2 [Credit card]: a few months ago i discovered an unauthorized charge on my credit card statement for i con...
Rank 3 [Credit card]: after having my card fraudulently used issues over refunds for fraud but not by tying them...


## 4. Cross-Encoder Re-Ranking Pipeline
Implementing a two-stage retrieval architecture:
- **Bi-Encoders (Stage 1):** Fast candidate generation (Top 10 from RRF).
- **Cross-Encoder (Stage 2):** Deep joint cross-attention scoring using `cross-encoder/ms-marco-MiniLM-L-6-v2`.
- Computes full token-to-token cross-attention between `[Query, Candidate Document]` pairs to resolve fine-grained semantic distinctions (e.g., Credit Card vs. Debit Card).

In [34]:
# 1. Cross-Encoder Model load karte hain (Trained on MS MARCO Search Dataset)
print("Loading Cross-Encoder re-ranker model ('ms-marco-MiniLM-L-6-v2')...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Re-Ranking Function
def hybrid_search_with_reranking(query, final_top_k=3, initial_candidates_count=10):
    # Step A: Pehle RRF se Top 10 acche candidates nikaalte hain
    candidates = hybrid_search_rrf(query, top_k=initial_candidates_count)
    
    # Step B: Query aur har candidate text ka pair (jodi) banate hain
    pairs = [[query, item['text']] for item in candidates]
    
    # Step C: Cross-Encoder se exact relevance score nikaalte hain
    cross_scores = reranker.predict(pairs)
    
    # Scores ko candidates ke sath jodte hain
    for idx, item in enumerate(candidates):
        item['cross_encoder_score'] = round(float(cross_scores[idx]), 4)
        
    # Step D: Cross-Encoder score ke mutabiq dobara sort (Re-rank) karte hain
    reranked_results = sorted(candidates, key=lambda x: x['cross_encoder_score'], reverse=True)[:final_top_k]
    
    # Rank update karte hain
    for new_rank, item in enumerate(reranked_results, 1):
        item['final_rank'] = new_rank
        
    return reranked_results

# 3. Test Run on the SAME Query:
test_query = "unauthorized transactions and fraudulent credit card fees disputed but denied"
reranked_results = hybrid_search_with_reranking(test_query, final_top_k=3)

print(f"\n --- CROSS-ENCODER RE-RANKED RESULTS for: \"{test_query}\" ---\n")
for res in reranked_results:
    print(f"Final Rank {res['final_rank']} [Cross-Encoder Score: {res['cross_encoder_score']}] - Category: {res['category'].upper()}")
    print(f"   Snippet: \"{res['text'][:150]}...\"\n")

Loading Cross-Encoder re-ranker model ('ms-marco-MiniLM-L-6-v2')...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2352.79it/s]



 --- CROSS-ENCODER RE-RANKED RESULTS for: "unauthorized transactions and fraudulent credit card fees disputed but denied" ---

Final Rank 1 [Cross-Encoder Score: 7.4719] - Category: BANK ACCOUNT OR SERVICE
   Snippet: "my debit card was lost stolen and many unauthorized transactions appeared on my account i disputed these charges and they were denied i called suntrus..."

Final Rank 2 [Cross-Encoder Score: 3.5487] - Category: BANK ACCOUNT OR SERVICE
   Snippet: "i disputed fraudulent transactions on my chase debit card they initially credited and to my checking account then reversed the credit they gave no exp..."

Final Rank 3 [Cross-Encoder Score: 2.4406] - Category: BANK ACCOUNT OR SERVICE
   Snippet: "my bank navy federal credit union has charged me for nsf return check fees totaling i called and disputed these fees as they were debit card transacti..."



In [32]:
# Inspect all 10 candidates and their Cross-Encoder scores
all_candidates = hybrid_search_rrf(test_query, top_k=10)
pairs = [[test_query, item['text']] for item in all_candidates]
scores = reranker.predict(pairs)

print(" --- ALL 10 CANDIDATES SCORED BY CROSS-ENCODER --- \n")
for i, item in enumerate(all_candidates):
    print(f"Candidate {i+1} [{item['category']}] -> Score: {scores[i]:.4f}")
    print(f"   Snippet: \"{item['text'][:120]}...\"\n")

 --- ALL 10 CANDIDATES SCORED BY CROSS-ENCODER --- 

Candidate 1 [Bank account or service] -> Score: 7.4719
   Snippet: "my debit card was lost stolen and many unauthorized transactions appeared on my account i disputed these charges and the..."

Candidate 2 [Bank account or service] -> Score: 3.5487
   Snippet: "i disputed fraudulent transactions on my chase debit card they initially credited and to my checking account then revers..."

Candidate 3 [Credit card] -> Score: -0.1692
   Snippet: "i lost my credit card and and their are fraudulent charges and transactions of the charges do n t belong to me i have ne..."

Candidate 4 [Credit reporting] -> Score: 1.9784
   Snippet: "i m a victim of identity theft my credit card has been lost stolen and compromised all of the charges on the account is ..."

Candidate 5 [Credit card] -> Score: 1.0951
   Snippet: "a few months ago i discovered an unauthorized charge on my credit card statement for i contacted the merchant regarding ..."

Candida

In [33]:
credit_query = "Citibank increased my credit card APR interest rate and lowered my credit limit without notice"
cc_results = hybrid_search_with_reranking(credit_query, final_top_k=3)

print(f"\n --- RESULTS FOR PURE CREDIT CARD QUERY ---\n")
for res in cc_results:
    print(f"Rank {res['final_rank']} [Score: {res['cross_encoder_score']}] - Category: {res['category'].upper()}")
    print(f"   Snippet: \"{res['text'][:140]}...\"\n")


 --- RESULTS FOR PURE CREDIT CARD QUERY ---

Rank 1 [Score: 5.2431] - Category: CREDIT CARD
   Snippet: "my citi dividend credit card apr was raised from to after citi had given me written notice to opt out of this increase the increase was due ..."

Rank 2 [Score: 5.1885] - Category: CREDIT CARD
   Snippet: "after being from status as a my credit card company chase bank card services decrease my credit limits and increase my apr rate with out not..."

Rank 3 [Score: 3.1116] - Category: CREDIT CARD
   Snippet: "capital one randomly and without my consent to reject has increased my credit limit three times in the last year these have been significant..."



> 📌 **Key Architectural Insight: Intent Alignment vs. Lexical Constraints in Cross-Encoders**
> 1. **Situational Alignment (Why Bank Account Scored 7.47):**
>    - In the query *"unauthorized transactions and fraudulent credit card fees disputed but denied"*, the Cross-Encoder awarded its highest score ($7.47$) to a `Bank account or service` narrative rather than a `Credit card` narrative.
>    - **Root Cause:** The Cross-Encoder prioritized the **complete grievance trajectory** (*unauthorized transactions $\to$ filed dispute $\to$ denied by bank*) over the specific payment instrument (*credit* vs *debit* card).
> 2. **Pre-training Bias:** The `ms-marco-MiniLM-L-6-v2` model is trained on search passage relevance where procedural matches are rewarded heavily. To a semantic model, debit card dispute denials and credit card dispute denials represent near-identical customer frustrations.
> 3. **Disambiguation via Exclusive Domain Tokens:** When queries contain features exclusive to credit cards (*APR rates, credit limits, balance transfers*), the Cross-Encoder cleanly isolates and ranks `Credit card` at Rank 1.
> 4. **Production Recommendation:** For enterprise deployment, if strict entity segregation is required, applying **Metadata Pre-Filtering** (e.g., filtering candidate pool by product tag before re-ranking) ensures both exact entity compliance and deep procedural relevance.